# Title embeddings

Run after `python -m similarity.embeddings`.

In [ ]:
# Work from the repo root, so `similarity` imports and the data/ paths resolve when opened from notebooks/
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

In [ ]:
import time
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
import similarity.search as s
from similarity.embeddings import TITLE_MODEL, rescale
from similarity.features import prepare_text
from similarity.preprocessing import load_products

products = load_products()
f = s.load_features()
e = s.title_embeddings(f)
e.model, e.vectors.shape, round(e.baseline, 3)

In [ ]:
s.METHOD_WEIGHTS

## the model

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(TITLE_MODEL, device="cpu")
v = model.encode(["Girls cotton frock"], normalize_embeddings=True)
v.shape, v[0, :8].round(3), np.linalg.norm(v[0]).round(3)

In [ ]:
row = 3
products.loc[row, "name"], e.vectors[row, :8].round(3)

## pairs: tfidf vs minilm

In [ ]:
tfidf = TfidfVectorizer(preprocessor=prepare_text, min_df=2).fit(products["name"])   # same settings as stage 02

pairs = [
    ("Girls cotton frock", "Girls cotton dress"),
    ("Women's straight kurta", "Women's straight kurti"),
    ("Girls cotton frok", "Girls cotton frock"),
    ("Men's cotton T-shirt", "Men's cotton tee"),
    ("Woolen beanie", "Winter knit cap"),
    ("Kalamkari dress material", "Kalamkari blouse fabric"),
    ("Kalamkari dress material", "Women's printed dress"),
    ("Polo T-shirt for men", "U.S. Polo Assn. men's shirt"),
    ("Girls cotton frock", "Men's leather wallet"),
]
left = model.encode([a for a, _ in pairs], normalize_embeddings=True)
right = model.encode([b for _, b in pairs], normalize_embeddings=True)
minilm = np.einsum("ij,ij->i", left, right)
words = (tfidf.transform([a for a, _ in pairs]).multiply(tfidf.transform([b for _, b in pairs]))).sum(axis=1).A1
pd.DataFrame({
    "a": [a for a, _ in pairs],
    "b": [b for _, b in pairs],
    "tfidf": words.round(2),
    "minilm": minilm.round(2),
    "minilm_rescaled": rescale(minilm, e.baseline).round(2),
})

## random pairs: why rescale

In [ ]:
rng = np.random.default_rng(0)
a, b = rng.choice(len(f.ids), 5000), rng.choice(len(f.ids), 5000)
spread = pd.DataFrame({
    "tfidf": f.text[a].multiply(f.text[b]).sum(axis=1).A1,
    "minilm": np.einsum("ij,ij->i", e.vectors[a], e.vectors[b]),
})
spread["minilm_rescaled"] = rescale(spread["minilm"].to_numpy(), e.baseline)
spread.describe(percentiles=[0.5, 0.9, 0.99]).round(3)

In [ ]:
(spread > 0).mean().round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3), sharey=True)
for ax, column in zip(axes, spread.columns):
    ax.hist(spread[column], bins=50, range=(-0.2, 1))
    ax.set_title(column)
axes[1].axvline(e.baseline, color="black", linestyle="--")
axes[0].set_ylabel("random pairs")
plt.tight_layout()

## one product, both methods

In [ ]:
def compare(uniq_id, k=10):
    row = f.row_of[uniq_id]
    print(products.loc[row, "name"])
    tops = {}
    for method in ["tfidf", "minilm"]:
        order, _ = s.rank(f, row, s.METHOD_WEIGHTS[method])
        tops[method] = products.loc[order[:k], "name"].str[:60].tolist()
    return pd.DataFrame(tops, index=range(1, k + 1))

In [ ]:
compare("2304753b72be21b48128f7d2ae482c9f")   # Q14 frock

In [ ]:
compare("0f8a3a26f3e563621eda640aa3ecd17e")   # Q15 kurta

In [ ]:
compare("26d41bdc1495de290bc8e6062d927729")   # Q13 kalamkari blouse fabric

In [ ]:
compare("8e724efa4103309725237750410b6a2a")   # Q18 "Frok"

## title only

In [ ]:
row = f.row_of["2304753b72be21b48128f7d2ae482c9f"]
title_only = {}
for name in ["text", "text_embedding"]:
    order, score = s.rank(f, row, {name: 1.0})
    title_only[name] = (products.loc[order[:10], "name"].str[:50] + "  " + score[:10].round(2).astype(str)).tolist()
pd.DataFrame(title_only, index=range(1, 11))

## breakdown

In [ ]:
order, _ = s.rank(f, row, s.METHOD_WEIGHTS["minilm"])
s.score_breakdown(f, row, order[0], s.METHOD_WEIGHTS["minilm"]).round(3)

In [ ]:
s.score_breakdown(f, row, order[0], s.METHOD_WEIGHTS["tfidf"]).round(3)

## timing

In [ ]:
for method in ["tfidf", "minilm"]:
    start = time.perf_counter()
    for r in range(100):
        s.rank(f, r, s.METHOD_WEIGHTS[method])
    print(method, round((time.perf_counter() - start) / 100 * 1000, 1), "ms per search")

## evaluation

`python -m evals.run_eval --alternative minilm` writes the workbook.

In [ ]:
pd.read_excel("evals/eval_results.xlsx", sheet_name="Methods")